# AI-sfa 나카무라 Wan LoRA 본 학습 2차 — Kaggle T4 (fp16)

학습 자료 2판(효과 없음 23조각 제외 + 인물·로봇 캡션 문장)으로 3000스텝 학습한다(학습 시험: 4.41초/스텝 → 약 3.7시간).
GPU 0이 학습하는 동안 GPU 1이 1000·2000·3000스텝 저장본으로 평가 영상(프롬프트 7 × 씨앗 3, 세기 0.8, CPU 노이즈)을 만든다 — 평가 3판 기준선과 같은 씨앗.

**설정:**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. **(필수 — 학습 자료가 비공개 Drive에 있음)** **Add-ons → Secrets**에 `RCLONE_GDRIVE_AUTH_B64`를 붙이면 결과가 Drive `AI-sfa/07_도구/PC_WanLoRA/`로 올라간다. 없으면 Output 탭에서 받는다.

**실행:** **Save Version → Save & Run All (Commit)**. 예상 5–6시간(추정). 결과 `nkmr_wan13_v2_out.zip`(LoRA 저장본 500스텝마다, 저장본 평가 시트 `ckpt_compare/`, 기록 `train/`).

In [ ]:
# 1. (선택) Drive 연결 — 비밀값이 없으면 건너뛴다. 인증값은 출력하지 않는다
import os, json, base64, pathlib, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    raw = UserSecretsClient().get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
    d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
    p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
    lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
    if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
    if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
    lines.append('token = ' + d['token'])
    (p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
    del raw, d, lines
    subprocess.run('cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/', shell=True, check=True)
    print('Drive 연결됨')
except Exception as e:
    print('Drive 연결 없이 진행(결과는 Output 탭):', type(e).__name__)

In [ ]:
# 2. 작업자 파일 쓰기 → 실행(받기·설치 → 캐시(GPU 2장) → GPU0 학습 + GPU1 저장본 평가 → zip)
open('/tmp/kaggle_wan_train_worker.py', 'w', encoding='utf-8').write("#!/usr/bin/env python3\n\"\"\"Kaggle T4 x2: 나카무라 Wan2.1 1.3B LoRA 본 학습 + 중간 저장본 평가 (2026-10-28 밤).\n\n학습 시험(10-28, data/style_kb/kaggle_wan_train_test_2026-10-28.json): T4 fp16 4.41 s/it, NaN 없음, 손실이 bf16과 같음 → 3000스텝 약 3.7시간.\n순서:\n  1) 받기·설치(평가 노트북과 같은 musubi 커밋·모델), 학습 자료 zip(--data-url)\n  2) 잠재 캐시를 GPU 2장에 반씩(VAE fp16 — 시험에서 bf16 에뮬레이션은 15.7초/조각), 글 캐시(T5 fp8 저장 + bf16)\n  3) GPU 0: fp16 학습(설정은 PC와 같음: LoRA 32/16, AdamW 2e-4, shift 3.0, checkpointing, 시드 42), --save-every마다 저장본을 모두 남김\n  4) GPU 1: 저장본이 나올 때마다(기본 1000·2000·3000) 평가 3판 프롬프트 7개 × 씨앗 3개를 세기 0.8·CPU 노이즈로 생성 → 같은 씨앗의 기준선과 비교\n  5) LoRA 저장본·평가 시트·기록 → zip → Drive(비밀값 있을 때)\n도우미는 kaggle_wan_eval_worker.py·kaggle_wan_eval3_worker.py(같은 브랜치)를 받아 쓴다. 서버에서는 GPU 실행을 못 했다(문법·명령 구성만 확인).\n사용(노트북): python kaggle_wan_train_worker.py --data-url URL [--name nkmr_wan13_v2] [--steps 3000] [--save-every 500] [--eval-steps 1000,2000,3000]\n\"\"\"\nimport argparse\nimport glob\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport zipfile\n\nW = os.environ.get(\"WAN_DIR\", \"/tmp/wan\")\nRAW = \"https://raw.githubusercontent.com/twoapple08/AI-sfa/ccr-bccf15a6-d6ee46/tools\"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--out\", default=\"/kaggle/working\")\n    ap.add_argument(\"--data-url\", required=True)\n    ap.add_argument(\"--name\", default=\"nkmr_wan13_v2\")\n    ap.add_argument(\"--steps\", type=int, default=3000)\n    ap.add_argument(\"--save-every\", type=int, default=500)\n    ap.add_argument(\"--eval-steps\", default=\"1000,2000,3000\")\n    ap.add_argument(\"--hours\", type=float, default=11.0, help=\"Kaggle 세션 12시간 — 이 시간이 지나면 학습을 멈추고 결과를 묶는다\")\n    a = ap.parse_args()\n    os.makedirs(W, exist_ok=True)\n    for f in (\"kaggle_wan_eval_worker.py\", \"kaggle_wan_eval3_worker.py\"):\n        subprocess.run([\"curl\", \"-sSL\", \"-o\", f\"{W}/{f}\", f\"{RAW}/{f}\"], check=True)\n    sys.path.insert(0, W)\n    import kaggle_wan_eval3_worker as e3\n    import kaggle_wan_eval_worker as ev\n    t_end = time.time() + a.hours * 3600\n    cap, ng, mt = ev.setup()  # 모델·LoRA(기존)·musubi 설치. T4 bf16 곱셈 확인 포함\n    import pc_wan_lora_run as pc\n    ev.set_t5_dtype(mt, \"bfloat16\")\n    m = os.path.join(W, \"models\")\n    dit = os.path.join(m, os.path.basename(ev.MODELS[\"dit_fp16\"][0]))\n    src = os.path.join(mt, \"src\", \"musubi_tuner\")\n    logdir = os.path.join(a.out, \"train\")\n    os.makedirs(logdir, exist_ok=True)\n\n    # 1) 자료: 조각을 반씩 두 폴더로(캐시를 GPU 2장에 나누기 위해), 학습 설정은 두 폴더를 함께\n    data = os.path.join(W, \"data_v2\")\n    if not os.path.exists(os.path.join(data, \"a\")):\n        zp = os.path.join(W, \"wanlora_v2.zip\")\n        if a.data_url.startswith(\"gdrive:\"):  # 비공개 Drive 파일 — 노트북 1번 셀의 rclone 연결(비밀값 RCLONE_GDRIVE_AUTH_B64) 필요\n            subprocess.run([\"rclone\", \"copyto\", a.data_url, zp], check=True)\n        else:\n            subprocess.run([\"curl\", \"-sSL\", \"--retry\", \"5\", \"-o\", zp, a.data_url], check=True)\n        with zipfile.ZipFile(zp) as z:\n            mp4 = sorted(n for n in z.namelist() if n.startswith(\"video/\") and n.endswith(\".mp4\"))\n            for k, n in enumerate(mp4):\n                half = \"a\" if k % 2 == 0 else \"b\"\n                os.makedirs(os.path.join(data, half, \"video\"), exist_ok=True)\n                for nn in (n, n[:-4] + \".txt\"):\n                    open(os.path.join(data, half, \"video\", os.path.basename(nn)), \"wb\").write(z.read(nn))\n            for nn in (\"captions.csv\",):\n                if nn in z.namelist():\n                    open(os.path.join(a.out, \"train\", \"captions_used.csv\"), \"wb\").write(z.read(nn))\n    general = '[general]\\nresolution = [512, 288]\\ncaption_extension = \".txt\"\\nbatch_size = 1\\nenable_bucket = false\\n'\n    ds = lambda h: f'\\n[[datasets]]\\nvideo_directory = \"{data}/{h}/video\"\\ncache_directory = \"{data}/{h}/cache\"\\nframe_extraction = \"full\"\\nmax_frames = 33\\n'  # noqa: E731\n    tomls = {h: os.path.join(data, f\"ds_{h}.toml\") for h in (\"a\", \"b\", \"all\")}\n    open(tomls[\"a\"], \"w\").write(general + ds(\"a\"))\n    open(tomls[\"b\"], \"w\").write(general + ds(\"b\"))\n    open(tomls[\"all\"], \"w\").write(general + ds(\"a\") + ds(\"b\"))\n    nclip = len(glob.glob(os.path.join(data, \"*\", \"video\", \"*.mp4\")))\n    res = {\"clips\": nclip, \"name\": a.name, \"steps\": a.steps}\n    ev.say(f\"학습 자료 {nclip}조각\")\n\n    # 2) 캐시\n    def cache_lat(h, gpu):\n        log = os.path.join(logdir, f\"cache_latents_{h}.log\")\n        t0 = time.time()\n        with open(log, \"w\") as fh:\n            rc = subprocess.run([sys.executable, os.path.join(src, \"wan_cache_latents.py\"), \"--dataset_config\", tomls[h], \"--vae\",\n                                 os.path.join(m, \"wan_2.1_vae.safetensors\"), \"--vae_dtype\", \"float16\", \"--vae_cache_cpu\"],\n                                cwd=mt, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=fh, stderr=subprocess.STDOUT).returncode\n        res[f\"cache_latents_{h}\"] = dict(rc=rc, min=round((time.time() - t0) / 60, 1))\n        ev.say(f\"잠재 캐시 {h}:\", res[f\"cache_latents_{h}\"])\n    ths = [threading.Thread(target=cache_lat, args=(h, g % max(1, ng))) for g, h in enumerate((\"a\", \"b\"))]\n    for t in ths:\n        t.start()\n    for t in ths:\n        t.join()\n    t0 = time.time()\n    with open(os.path.join(logdir, \"cache_text.log\"), \"w\") as fh:\n        rc = subprocess.run([sys.executable, os.path.join(src, \"wan_cache_text_encoder_outputs.py\"), \"--dataset_config\", tomls[\"all\"],\n                             \"--t5\", os.path.join(m, \"models_t5_umt5-xxl-enc-bf16.pth\"), \"--batch_size\", \"8\", \"--fp8_t5\"],\n                            cwd=mt, env=dict(os.environ, CUDA_VISIBLE_DEVICES=\"0\"), stdout=fh, stderr=subprocess.STDOUT).returncode\n    res[\"cache_text\"] = dict(rc=rc, min=round((time.time() - t0) / 60, 1))\n    ev.say(\"글 캐시:\", res[\"cache_text\"])\n    if any(v.get(\"rc\") for k, v in res.items() if k.startswith(\"cache\")):\n        raise SystemExit(\"캐시 실패 — train/ 기록을 Claude에게\")\n\n    # 3) 학습(GPU 0) + 4) 저장본 평가(GPU 1)\n    out = os.path.join(W, \"out_v2\")\n    os.makedirs(out, exist_ok=True)\n    tlog = os.path.join(logdir, \"train.log\")\n    cmd = [sys.executable, \"-m\", \"accelerate.commands.launch\", \"--num_cpu_threads_per_process\", \"1\", \"--num_processes\", \"1\", \"--mixed_precision\", \"fp16\",\n           os.path.join(src, \"wan_train_network.py\"), \"--task\", \"t2v-1.3B\", \"--dit\", dit, \"--dataset_config\", tomls[\"all\"],\n           \"--sdpa\", \"--mixed_precision\", \"fp16\", \"--optimizer_type\", \"AdamW\", \"--learning_rate\", \"2e-4\",\n           \"--gradient_checkpointing\", \"--max_data_loader_n_workers\", \"2\", \"--persistent_data_loader_workers\",\n           \"--network_module\", \"networks.lora_wan\", \"--network_dim\", \"32\", \"--network_alpha\", \"16\",\n           \"--timestep_sampling\", \"shift\", \"--discrete_flow_shift\", \"3.0\", \"--max_train_steps\", str(a.steps),\n           \"--save_every_n_steps\", str(a.save_every), \"--seed\", \"42\", \"--output_dir\", out, \"--output_name\", a.name]\n    done = threading.Event()\n    eval_steps = [int(x) for x in a.eval_steps.split(\",\") if x]\n    names = [\"0\", \"1\", \"2\", \"3\", \"1v\", \"3v\", \"4r\"]\n    prompts = {str(i): p for i, p in enumerate(pc.PROMPTS)}\n    prompts.update(e3.EXTRA)\n    sd = os.path.join(W, \"samples_ckpt\")\n    os.makedirs(sd, exist_ok=True)\n    items = [(f\"{n}_s{s}\", prompts[n], 1000 * (k + 1) + s) for k, n in enumerate(names) for s in e3.SEEDS]  # 평가 3판과 같은 씨앗\n\n    def evaluator():\n        gpu = 1 if ng > 1 else 0\n        lock = threading.Lock()\n        pending = list(eval_steps)\n        while pending:\n            st = pending[0]\n            cands = [os.path.join(out, f\"{a.name}-step{st:08d}.safetensors\")]\n            if st >= a.steps and done.is_set():  # 마지막 스텝은 최종 파일 이름으로만 남을 수도 있다\n                cands.append(os.path.join(out, f\"{a.name}.safetensors\"))\n            f = next((c for c in cands if os.path.exists(c)), cands[0])\n            if os.path.exists(f):\n                s1 = os.path.getsize(f)\n                time.sleep(15)\n                if os.path.getsize(f) != s1:\n                    continue\n                lora = pc.merge_ready(f, dit)\n                ev.say(f\"저장본 {st}스텝 평가 시작(GPU {gpu})\")\n                try:\n                    ev.run_group(dict(tag=f\"st{st}\", mult=0.8, neg=None, items=items, dit=dit, lora=lora, mt=mt, sd=sd, logdir=logdir, cpu_noise=True), gpu, lock)\n                except Exception as e:  # 평가 실패가 학습을 멈추지 않게\n                    ev.say(\"평가 실패:\", e)\n                pending.pop(0)\n            elif done.is_set():\n                ev.say(f\"저장본 {st}스텝 없음 — 평가 건너뜀\")\n                pending.pop(0)\n            else:\n                time.sleep(60)\n\n    eth = threading.Thread(target=evaluator)\n    eth.start()\n    t0 = time.time()\n    left = t_end - time.time() - 45 * 60  # 마지막 평가·묶기에 45분\n    ev.say(f\"학습 시작 {a.steps}스텝(fp16, 예상 {a.steps * 4.41 / 3600:.1f}시간), 시간 한도 {left / 3600:.1f}시간\")\n    with open(tlog, \"w\", encoding=\"utf-8\") as fh:\n        p = subprocess.Popen(cmd, cwd=mt, env=dict(os.environ, CUDA_VISIBLE_DEVICES=\"0\", PYTHONUNBUFFERED=\"1\"),\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors=\"replace\")\n        last = 0\n        for line in p.stdout:\n            fh.write(line)\n            if (\"steps:\" in line and time.time() - last > 300) or \"Error\" in line or \"Traceback\" in line:\n                ev.say(line.rstrip()[-140:])\n                last = time.time()\n            if time.time() - t0 > left:\n                ev.say(\"시간 한도 — 학습을 멈춤(저장본까지만)\")\n                p.kill()\n                break\n        p.wait()\n    done.set()\n    res[\"train\"] = dict(rc=p.returncode, min=round((time.time() - t0) / 60, 1))\n    ev.say(\"학습 끝:\", res[\"train\"])\n    eth.join()\n\n    # 5) 시트·묶기\n    base_dir = os.path.join(a.out, \"ckpt_compare\")\n    conds = [f\"st{s}\" for s in eval_steps]\n    e3.sheets(ev, sd, base_dir, names, conds)\n    loras = sorted(glob.glob(os.path.join(out, f\"{a.name}*.safetensors\")))\n    loras = [f for f in loras if not f.endswith(\"_merge.safetensors\")]\n    res[\"loras\"] = [os.path.basename(f) for f in loras]\n    json.dump(res, open(os.path.join(logdir, \"summary.json\"), \"w\", encoding=\"utf-8\"), ensure_ascii=False, indent=1)\n    zp = os.path.join(a.out, f\"{a.name}_out.zip\")\n    with zipfile.ZipFile(zp, \"w\", zipfile.ZIP_STORED) as z:\n        for f in loras:\n            z.write(f, \"lora/\" + os.path.basename(f))\n        for f in sorted(glob.glob(os.path.join(sd, \"*.mp4\"))):\n            z.write(f, \"samples_ckpt/\" + os.path.basename(f))\n        for d in (\"ckpt_compare\", \"train\"):\n            for f in sorted(glob.glob(os.path.join(a.out, d, \"*\"))):\n                z.write(f, os.path.relpath(f, a.out))\n    ev.say(f\"완료 → {zp} ({os.path.getsize(zp) / 1e9:.2f}GB)\", json.dumps(res, ensure_ascii=False))\n    if shutil.which(\"rclone\") and os.path.exists(os.path.expanduser(\"~/.config/rclone/rclone.conf\")):\n        r = subprocess.run([\"rclone\", \"copy\", zp, \"gdrive:AI-sfa/07_도구/PC_WanLoRA/\"], capture_output=True, text=True)\n        ev.say(\"Drive 업로드:\", \"성공 → AI-sfa/07_도구/PC_WanLoRA/\" if r.returncode == 0 else \"실패 — Output 탭에서 받아 주세요\")\n\n\nif __name__ == \"__main__\":\n    main()\n")
!python -u /tmp/kaggle_wan_train_worker.py --out /kaggle/working --data-url 'gdrive:AI-sfa/07_도구/PC_WanLoRA/wanlora_v2.zip'